# Llama 3.2-1B QLoRA 샘플 테스트 (5,000행)
- 전체 데이터에서 **5,000행만 샘플링**해서 빠르게 결과 확인
- T4 기준 예상 소요: **약 15~20분**
- 파이프라인 정상 동작 및 응답 품질 미리보기용

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택

## 1. 패키지 설치

In [16]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets sentence-transformers scikit-learn

<IPython.core.display.Javascript object>

In [17]:
# Colab 세션 자동 유지 (끊김 방지)
import time, threading
from IPython.display import display, Javascript

def _keep_alive():
    while True:
        time.sleep(55)
        try:
            display(Javascript('''
                (() => {
                    const btn = document.querySelector("colab-connect-button");
                    if (btn) btn.click();
                    console.log("세션 유지:", new Date().toLocaleTimeString());
                })();
            '''))
        except Exception:
            pass

threading.Thread(target=_keep_alive, daemon=True).start()
print('✅ 세션 유지 스크립트 실행됨')

✅ 세션 유지 스크립트 실행됨


## 2. HuggingFace 로그인

In [ ]:
from huggingface_hub import login
import os
login(token=os.environ.get("HF_TOKEN"))  # set HF_TOKEN env var before running

## 3. 구글 드라이브 마운트 & 데이터 업로드

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
import shutil
shutil.copy('/content/drive/MyDrive/aiproject/mountain/merged_train.csv', 'merged_train.csv')
shutil.copy('/content/drive/MyDrive/aiproject/mountain/merged_val.csv',   'merged_val.csv')
print('데이터 복사 완료')

데이터 복사 완료


## 4. 임포트

In [21]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print('torch version:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — GPU 런타임으로 변경 필요')

torch version: 2.10.0+cu128
GPU: Tesla T4


## 5. 설정

In [ ]:
MODEL_ID      = 'meta-llama/Llama-3.2-1B-Instruct'
SAVE_PATH     = '/content/drive/MyDrive/aiproject/mountain/llama32_1b_sample'
MAX_LENGTH    = 512
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
EPOCHS        = 1       # 샘플 테스트는 1 epoch
LR            = 2e-4
TRAIN_SAMPLE  = 5000   # ← 여기서 샘플 수 조절
VAL_SAMPLE    = 500

print(f'train 샘플: {TRAIN_SAMPLE:,}행 | val 샘플: {VAL_SAMPLE:,}행')

train 샘플: 5,000행 | val 샘플: 500행


## 6. 데이터 로드 및 샘플링

In [32]:
def load_data(train_path='merged_train.csv', val_path='merged_val.csv'):
    train_df = pd.read_csv(train_path, encoding='utf-8-sig')
    val_df   = pd.read_csv(val_path,   encoding='utf-8-sig')

    # 전체 데이터에서 TRAIN_SAMPLE / VAL_SAMPLE 만큼만 샘플링
    if len(train_df) > TRAIN_SAMPLE:
        train_df = train_df.sample(n=TRAIN_SAMPLE, random_state=42)
    if len(val_df) > VAL_SAMPLE:
        val_df = val_df.sample(n=VAL_SAMPLE, random_state=42)

    return Dataset.from_pandas(train_df.reset_index(drop=True)), \
           Dataset.from_pandas(val_df.reset_index(drop=True))


def format_prompt(row):
    return {
        'text': (
            f"<|begin_of_text|>"
            f"<|start_header_id|>user<|end_header_id|>\n{row['question']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n{row['answer']}<|eot_id|>"
        )
    }

# 토큰화 함수 정의 (tokenizer와 MAX_LENGTH는 이미 전역 변수로 정의되어 있음)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

train_ds, val_ds = load_data()
train_ds = train_ds.map(format_prompt)
val_ds   = val_ds.map(format_prompt)

# 데이터셋을 미리 토큰화
train_ds = train_ds.map(tokenize_function, batched=True, remove_columns=train_ds.column_names)
val_ds   = val_ds.map(tokenize_function, batched=True, remove_columns=val_ds.column_names)

print(f'train: {len(train_ds):,}행  |  val: {len(val_ds):,}행')
print('\n[샘플]')
print(tokenizer.decode(train_ds[0]['input_ids'][:300]))

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

train: 5,000행  |  val: 500행

[샘플]
<|begin_of_text|><|begin_of_text|><|start_header_id|>user<|end_header_id|>
턱관절 탈구를 피하기 위해 일상 생활에서 어떤 습관을 가지는 것이 좋을까요?<|eot_id|><|start_header_id|>assistant<|end_header_id|>
턱관절 탈구 환자를 위한 식이요법은 턱관절 문제를 최소화하고 식사 시 발생할 수 있는 불편함을 완화하는 데 도움이 됩니다. 턱관절에 부담을 줄 수 있는 과도한 씹기나 턱을 지속적으로 움직이는 습관을 피하고, 부드러운 음식을 섭취하는 것이 중요합니다. 또한, 염증을 줄일 수 있는 식품을 선택해야 합니다. 생선, 과일, 채소, 고구마, 닭고기, 녹차, 우유 등이 턱관절 건강에 도움이 되는 음식입니다. 턱관절 탈구로 인해 식사가 불편할 수 있으므로 염증을 줄일 수 있는 식품을 선택해야 합니다. 부드러운 음식을 섭취하면 턱관절에 무리를 주지 않고 소화가 잘 되므로 식사가 편해질 수 있습니다. 또한, 식이섬유가 풍부한 과일과 채소를 먹으면 턱 주변의 붓기를 줄일 수 있습니다. 또한, 녹차, 우유, 견과류, 요구르트도 염증을


## 7. 모델 로드 (4-bit 양자화)

In [24]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model)
print('모델 로드 완료')

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

모델 로드 완료


## 8. LoRA 설정

In [25]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


## 9. 학습

5,000행 × 1 epoch 기준 T4에서 약 15~20분 예상

In [33]:
from trl import SFTTrainer, SFTConfig

# TRL 라이브러리 업데이트 (max_seq_length 인자 오류 해결을 위해) - 이미 최신 버전으로 확인되었으나, 혹시 모를 경우를 대비해 유지
!pip install --upgrade trl

training_args = SFTConfig(
    output_dir=SAVE_PATH,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=False,
    bf16=True,
    logging_steps=20,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=1,
    load_best_model_at_end=True,
    report_to='none',
    # dataset_text_field는 데이터셋이 이미 토큰화되었으므로 제거
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    # max_seq_length는 데이터셋이 이미 토큰화되었으므로 제거
)

trainer.train()
trainer.save_model(SAVE_PATH)
print(f'모델 저장 완료: {SAVE_PATH}')

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Step,Training Loss,Validation Loss
100,1.032067,1.031589
200,0.962718,0.966862
300,0.961524,0.948304


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

모델 저장 완료: /content/drive/MyDrive/aiproject/mountain/llama32_1b_sample


## 10. RAG 설정 (DistilKoBERT 벡터 DB 구축)

샘플 5,000행 기준으로 벡터 DB를 구성합니다.

In [35]:
!pip install -q sentencepiece

from transformers import AutoTokenizer as BertTokenizer, AutoModel as BertModel
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import numpy as np
import os

RAG_MODEL      = 'monologg/distilkobert'
EMBEDDING_PATH = '/content/drive/MyDrive/aiproject/mountain/embeddings_sample.npy'

# sentencepiece 설치 후 다시 로드
bert_tokenizer = BertTokenizer.from_pretrained(RAG_MODEL, trust_remote_code=True)
bert_model     = BertModel.from_pretrained(RAG_MODEL).to(model.device)
bert_model.eval()

rag_df = pd.read_csv('merged_train.csv', encoding='utf-8-sig')[['question', 'answer']].dropna().reset_index(drop=True)
if len(rag_df) > TRAIN_SAMPLE:
    rag_df = rag_df.sample(n=TRAIN_SAMPLE, random_state=42).reset_index(drop=True)

def get_bert_embedding(text):
    inputs = bert_tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = bert_model(**inputs)
    return out.last_hidden_state[:, 0, :].cpu().numpy()[0]

if os.path.exists(EMBEDDING_PATH):
    question_embeddings = np.load(EMBEDDING_PATH)
    print(f'임베딩 로드 완료: {question_embeddings.shape}')
else:
    print('질문 벡터화 중...')
    question_embeddings = np.array([get_bert_embedding(q) for q in tqdm(rag_df['question'], desc='벡터화')])
    np.save(EMBEDDING_PATH, question_embeddings)
    print(f'임베딩 저장 완료: {question_embeddings.shape}')

def retrieve_context(user_question, top_k=2):
    user_emb = get_bert_embedding(user_question).reshape(1, -1)
    sims     = cosine_similarity(user_emb, question_embeddings)[0]
    top_idx  = sims.argsort()[-top_k:][::-1]
    return '\n\n'.join(
        f"Q: {rag_df.iloc[i]['question']}\nA: {rag_df.iloc[i]['answer']}"
        for i in top_idx
    )

print('✅ RAG 설정 완료')

tokenization_kobert.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/monologg/distilkobert:
- tokenization_kobert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_78b3253a26.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

질문 벡터화 중...


벡터화:   0%|          | 0/5000 [00:00<?, ?it/s]

임베딩 저장 완료: (5000, 768)
✅ RAG 설정 완료


## 11. 추론 테스트 (RAG + Llama)

DistilKoBERT로 유사 응급 사례를 먼저 검색하고,  
Llama가 그 내용을 참고해서 응급처치 안내를 생성합니다.

In [36]:
def test_inference(question: str) -> str:
    context = retrieve_context(question, top_k=2)

    prompt = (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n"
        f"당신은 산악 응급처치 전문 AI입니다. 아래 참고 정보를 바탕으로 응급처치 방법을 안내하세요.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"[참고 정보]\n{context}\n\n[질문]\n{question}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)


# 테스트 질문 3개
questions = [
    '등산 중 발목을 삐었을 때 응급처치 방법은?',
    '저체온증 증상이 나타나면 어떻게 해야 하나요?',
    '숨이 안 쉬어져요',
]

for q in questions:
    print(f'\n[질문] {q}')
    print(f'[답변] {test_inference(q)}')
    print('-' * 60)


[질문] 등산 중 발목을 삐었을 때 응급처치 방법은?
[답변] system
당신은 산악 응급처치 전문 AI입니다. 아래 참고 정보를 바탕으로 응급처치 방법을 안내하세요.user
[참고 정보]
Q: 발목을 자주 삐끗하여 발목 염좌 치료를 받았을 때, 테이핑을 시행하면 어떤 효과가 있을까요?
A: 염좌를 예방하기 위한 다른 방법들도 있습니다. 고관절 염좌를 예방하기 위해 다음과 같은 방법들을 고려할 수 있습니다. 1. 충분한 스트레칭과 근력 훈련을 실시하세요. 이를 통해 근육을 강화하고 관절의 안정성을 높일 수 있습니다. 2. 적절한 보호장비를 사용하세요. 운동 중에는 보호장구를 착용하고, 지나치게 강하게 운동하지 않도록 주의하세요. 3. 손목을 보호하기 위해 스플린트나 스트링이 있는 보호장비를 사용하세요. 4. 균형을 위해 균형 잡기 연습을 하세요. 5. 스트레스를 관리하기 위해 요가나 명상 등의 심신 안정 방법을 사용하세요. 염좌 예방을 위해서는 근력 훈련과 보호장비 사용, 적절한 보호장구 착용, 균형 잡기 연습 등 다양한 방법을 고려할 수 있습니다. 이러한 예방 조치들을 적용하여 부상의 위험을 최소화하세요.

Q: 발목을 자주 삐끗하여 발목 염좌 치료를 받았을 때, 테이핑을 하고 있는 것이 좋을까요?
A: 염좌를 예방하기 위해 일상적으로 주의할 점은 다음과 같습니다. 손상을 방지하기 위해서는 손목 보호 장비를 사용하는 것이 매우 중요합니다. 스노우보딩이나 인라인 스케이팅과 같이 관절을 많이 사용하는 운동을 할 때에는 손목 보호 장비를 사용하는 것이 좋습니다. 뿐만 아니라 스트레칭과 강화 운동을 꾸준히 하는 것도 중요합니다. 만약 갑작스럽게 과도한 운동을 하게 된다면 염좌가 발생할 수 있습니다. 따라서 운동을 할 때에는 관절을 보호하기 위해 적절한 보호대를 착용하는 것이 좋습니다. 마지막으로, 운동과 일상 생활의 균형을 유지하는 것이 중요합니다. 일상 생활에서는 관절을 안정적으로 지탱하는 스트레칭과 강화 운동을 지속적으로 실시해야 합니다. 관절을 보호하고 적

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[답변] system
당신은 산악 응급처치 전문 AI입니다. 아래 참고 정보를 바탕으로 응급처치 방법을 안내하세요.user
[참고 정보]
Q: 뇌진탕 증상이 악화되고 있는지 어떻게 알 수 있을까요?
A: 혼란, 두통 악화, 구토 또는 행동 변화와 같은 증상이 나타나거나 심해지면 뇌진탕이 악화되고 있을 가능성이 있습니다. 이러한 증상이 나타나면 즉시 의료진의 진료를 받아야 합니다.

Q: 비출혈이 자주 나타나는 증상은 무엇인가요?
A: 비출혈은 일상적인 활동 중에도 발생할 수 있으며, 주로 코피라고 알려진 비출혈로 알려져 있습니다. 비출혈은 주로 갑작스러운 출혈로 알려져 있으며, 다양한 원인으로 인해 발생할 수 있습니다. 비출혈은 주로 코피가 동반되는 경우가 많으며, 코의 기능적 이상이나 기타 질환과 연관될 수도 있습니다. 비출혈은 주로 코의 점막 손상이나 알레르기 반응, 비강 종양, 혈액질환, 혈관 질환 등과 관련이 있습니다. 또한, 비출혈은 비강의 내부에서 발생하는 경우도 있습니다. 비출혈이 발생할 경우, 즉시 의료진을 찾아 치료받는 것이 중요합니다. 비출혈의 원인과 심각성에 따라 적절한 치료 방법을 선택하고, 코를 부드럽게 다루는 것이 좋습니다. 특히, 습도가 낮은 환경이나 기온이 낮은 곳에 장기간 노출되는 것은 비출혈을 유발할 수 있으므로 주의가 필요합니다.

[질문]
저체온증 증상이 나타나면 어떻게 해야 하나요?assistant
저체온증은 체온이 낮아지거나 체온이 높아지면서 발생하는 질환입니다. 이러한 증상이 나타나면 즉시 의료진의 진료를 받아야 합니다. 저체온증은 체온이 낮아지거나 체온이 높아지면서 발생하는 질환으로, 이는 체온이 낮아지거나 체온이 높아지면서 발생하는 증상입니다. 이러한 증상이 나타나면 즉시 의료진의 진료를 받아야 합니다. 저체온증은 체온이 낮아지거나 체온이 높아지면서 발생하는 질환입니다. 이러한 증상이 나타나면 즉시 의료진의 진료를 받아야 합니다. 저체온증은 체온이 낮아지거나 체온이 높아지면서 발생하는 질환으로, 이는 체온이 낮아지거

In [37]:
# 테스트 질문 3개
questions = [
    '배에서 피가나요 어떡하죠??',
    '다리가 너무 아파요!!',
    '숨이 안 쉬어져요',
]

for q in questions:
    print(f'\n[질문] {q}')
    print(f'[답변] {test_inference(q)}')
    print('-' * 60)


[질문] 배에서 피가나요 어떡하죠??


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[답변] system
당신은 산악 응급처치 전문 AI입니다. 아래 참고 정보를 바탕으로 응급처치 방법을 안내하세요.user
[참고 정보]
Q: 혼자 있을 때 코피가 나면 어떻게 해야 할까요?
A: 환자는 앉아서 몸을 앞으로 숙이고 한 손으로 콧구멍을 10분 동안 막고 휴대폰이나 시계를 이용해 시간을 재야 합니다. 출혈이 멈추면 휴식을 취하고 코를 풀지 마십시오. 출혈이 다시 시작되면 위의 과정을 반복하거나, 멈추지 않으면 의료기관의 도움을 받으십시오.

Q: 집에서 혼자 가시를 빼는 것이 안전할까요?
A: 네, 적절한 소독 절차를 준수하고 상처를 조심스럽게 다룬다면 집에서 스스로 가시를 제거하는 것은 안전합니다. 작은 가시의 경우, 소독된 핀셋이나 집게를 사용하여 가시를 제거하고 상처를 깨끗이 닦은 후 접착 드레싱을 붙이세요. 하지만 가시가 크거나 깊숙이 박힌 경우, 또는 스스로 제거하는 것이 확실하지 않다면 의료 전문가의 도움을 받는 것이 좋습니다.

[질문]
배에서 피가나요 어떡하죠??assistant
배에서 피가 나면, 피를 제거하기 위해서는 다음과 같은 방법을 사용하세요. 피가 나면 피를 제거하기 위해 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서는 피를 제거하는 방법을 선택하세요. 피가 나면 피를 제거하기 위해서
------------------